In [1]:
from kaggle_secrets import UserSecretsClient

In [4]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("hugging-face-access-token")


In [6]:
!pip install -q transformers datasets peft accelerate sentencepiece

In [7]:
from huggingface_hub import login
login(secret_value_0)

### Loading Model 

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "meta-llama/Llama-2-7b-hf"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

In [9]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,194,304 || all params: 6,742,609,920 || trainable%: 0.0622


In [10]:
from datasets import load_dataset

dataset = load_dataset("glue", "sst2")

def format_example(example):
    example["prompt"] = f"Review: {example['sentence']}\nSentiment:"
    return example

dataset = dataset.map(format_example)

README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [21]:
def tokenize(example):
    label_text = "Positive" if example["label"] == 1 else "Negative"

    prompt = example["prompt"]
    full_text = prompt + " " + label_text

    # Tokenize full sequence
    tokenized = tokenizer(
        full_text,
        truncation=True,
        padding="max_length",
        max_length=128
    )

    # Tokenize prompt SAME WAY
    prompt_ids = tokenizer(
        prompt,
        truncation=True,
        max_length=128
    )["input_ids"]

    labels = tokenized["input_ids"].copy()

    prompt_len = len(prompt_ids)

    # Prevent full masking
    prompt_len = min(prompt_len, len(labels) - 1)

    labels[:prompt_len] = [-100] * prompt_len

    # Safety check
    if all(l == -100 for l in labels):
        labels[-1] = tokenized["input_ids"][-1]

    tokenized["labels"] = labels

    return tokenized

In [19]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=50,
    fp16=True,
    report_to="none"
)

In [22]:
tokenized_dataset = dataset.map(
    tokenize,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [23]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"]
)

trainer.train()

Step,Training Loss
50,0.000000
100,0.000000
150,0.000000
200,0.000000


KeyboardInterrupt: 